In [134]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# para procesamiento de texto
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk import RegexpTokenizer
from nltk.corpus import stopwords
from wordcloud import WordCloud

# para lematización en español
import simplemma

# para vectorización de texto
from sklearn.feature_extraction.text import CountVectorizer


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\diegklidia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\diegklidia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\diegklidia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [82]:
text = pd.read_excel(r'..\Data\textos_ODS.xlsx')
text

,textos,ODS
0,"""Aprendizaje"" y ""educación"" se consideran sinó...",4
1,No dejar clara la naturaleza de estos riesgos ...,6
2,"Como resultado, un mayor y mejorado acceso al ...",13
3,Con el Congreso firmemente en control de la ju...,16
4,"Luego, dos secciones finales analizan las impl...",5
...,...,...
9651,Esto implica que el tiempo de las mujeres en e...,5
9652,"Sin embargo, estas fallas del mercado implican...",3
9653,El hecho de hacerlo y cómo hacerlo dependerá e...,9
9654,"Esto se destacó en el primer estudio de caso, ...",6


In [83]:
text["ODS"].value_counts().sort_index()

ODS
1      505
2      369
3      894
4     1025
5     1070
6      695
7      787
8      446
9      343
10     352
11     607
12     312
13     464
14     377
15     330
16    1080
Name: count, dtype: int64

In [85]:
def limpieza_df (df):
    
    text.columns = df.columns.str.upper()
    text.drop_duplicates(inplace=True)
    text.dropna(inplace=True)

    return text

In [86]:
text_limp=limpieza_df(text)
text_limp["TEXTOS"][0]


'"Aprendizaje" y "educación" se consideran sinónimos de escolarización formal. Las organizaciones auxiliares, como las editoriales de educación, las juntas examinadoras y las organizaciones de formación de docentes, se consideran extensiones de los acuerdos establecidos por los gobiernos. Este marco de comprensión se ha vuelto cada vez más inadecuado.'

In [98]:
def nube_palabras(textos):
        
        cloud_text = ' '.join(word for text in textos for word in text)

        wordcloud = WordCloud(
            width=800,
            height=400,
            background_color ='white',
            min_font_size=10,
            max_font_size=110
        ).generate(cloud_text)
        plt.figure(figsize=(10, 10))
        plt.imshow(wordcloud)
        plt.axis('off')
        plt.show()


In [179]:
def preparacion_texto(df, cloud=False):
    # Handle both Series and individual strings
    if isinstance(df, str):
        df = pd.Series([df])
    
    # estandarización de texto a minúsculas
    df = df.str.lower()
    # tokenización de texto en palabras y eliminación de signos de puntuación
    tokenizer = RegexpTokenizer(r'\w+')
    tokenizer_no_punct = df.apply(lambda x: tokenizer.tokenize(x))
    # eliminación de stopwords
    nltk_stopwords = set(stopwords.words('spanish'))
    text_no_stopwords = tokenizer_no_punct.apply(lambda x: [word for word in x if word not in nltk_stopwords])
    # lematización de palabras
    text_lema = [[simplemma.lemmatize(token, lang='es') for token in text] for text in text_no_stopwords]
    if cloud:
        nube_palabras(text_lema)

    # Convert list to pandas Series first
    text_proce = pd.Series(text_lema).apply(lambda x: ' '.join(x))

    return text_proce[0] if len(text_proce) == 1 else text_proce


In [180]:
text_limp["TEXTOS"].str.lower()

0       "aprendizaje" y "educación" se consideran sinó...
1       no dejar clara la naturaleza de estos riesgos ...
2       como resultado, un mayor y mejorado acceso al ...
3       con el congreso firmemente en control de la ju...
4       luego, dos secciones finales analizan las impl...
                              ...                        
9651    esto implica que el tiempo de las mujeres en e...
9652    sin embargo, estas fallas del mercado implican...
9653    el hecho de hacerlo y cómo hacerlo dependerá e...
9654    esto se destacó en el primer estudio de caso, ...
9655    aunque existen programas para convertirse espe...
Name: TEXTOS, Length: 9656, dtype: object

In [181]:
text_prepa=preparacion_texto(text_limp["TEXTOS"])
print(text_limp["TEXTOS"][0])
text_prepa[0]

"aprendizaje" y "educación" se consideran sinónimos de escolarización formal. las organizaciones auxiliares, como las editoriales de educación, las juntas examinadoras y las organizaciones de formación de docentes, se consideran extensiones de los acuerdos establecidos por los gobiernos. este marco de comprensión se ha vuelto cada vez más inadecuado.


'aprendizaje educación considerar sinónimo escolarización formal organización auxiliar editorial educación junta examinador organización formación docente considerar extensión acuerdo establecer gobierno marco comprensión volver cada vez inadecuado'

In [173]:
text_limp["TEXTOS"]

0       "aprendizaje" y "educación" se consideran sinó...
1       no dejar clara la naturaleza de estos riesgos ...
2       como resultado, un mayor y mejorado acceso al ...
3       con el congreso firmemente en control de la ju...
4       luego, dos secciones finales analizan las impl...
                              ...                        
9651    esto implica que el tiempo de las mujeres en e...
9652    sin embargo, estas fallas del mercado implican...
9653    el hecho de hacerlo y cómo hacerlo dependerá e...
9654    esto se destacó en el primer estudio de caso, ...
9655    aunque existen programas para convertirse espe...
Name: TEXTOS, Length: 9656, dtype: object

In [ ]:

vectorizer = CountVectorizer(preprocessor=preparacion_texto)
x_train = vectorizer.fit_transform(text_limp["TEXTOS"])



In [ ]:
x_train

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (1, 22463)>